# Workbook 16 — SMS Customer Activation Analytics

## Customer Activation Intelligence System™

This workbook builds a customer activation intelligence layer for the Pizza House Grand Reopening SMS campaign.

The goal is to connect campaign delivery data, SimpleTexting contact exports, customer reply behavior, verified engagement signals, opt-outs, and future marketing segmentation into one structured analytics system.

Workbook 16 now uses the SimpleTexting Contacts Export as the canonical reply source. Screenshot and OCR workflows are no longer the primary architecture.

## 16.0 Business Context

Pizza House moved to a new location at 5050 Stockton Blvd and launched a Grand Reopening SMS campaign using SimpleTexting.

Earlier Pizza House workbooks focused on orders, revenue, demand timing, customer records, and Tableau reporting. Workbook 16 focuses on customer activation.

The campaign data includes structured delivery reports and SimpleTexting contact exports. The contact export contains customer phone numbers, imported address fields, opt-in status, unsubscribe indicators, and full Inbox conversation history.

The final objective is to build a reusable marketing database that identifies delivered messages, verified customer responses, opt-outs, invalid numbers, duplicate contacts, and future marketing-ready customers.

## 16.1 Customer Activation Framework

Workbook 16 follows a customer activation funnel:

```text
Customer List
    ↓
SMS Delivery
    ↓
SimpleTexting Contact Export
    ↓
Inbox Conversation Parsing
    ↓
Verified Customer Responses
    ↓
Phone-Based Customer Match
    ↓
Future Marketing Audience
```

The key business metric is **Verified Customer Responses**, not only exact `YES` replies.

Because coupon responses were delayed, some customers replied multiple times or used signals such as thumbs up, positive emojis, and HELP messages while waiting for the coupon. These are treated as verified engagement when the customer intent is clearly positive.

## 16.2 Setup — Imports and File Paths

This section defines the project folders used throughout Workbook 16.

The notebook follows the existing Pizza House repository structure and keeps raw source files, cleaned datasets, and final exports separated.

In [96]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import duckdb

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)

PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
CLEANED_DIR = DATA_DIR / "cleaned"
EXPORT_DIR = DATA_DIR / "exports"

SMS_RAW_DIR = RAW_DIR / "sms"

SC_DIR = SMS_RAW_DIR / "campaign_summaries"
DR_DIR = SMS_RAW_DIR / "delivery_reports"
CONTACTS_DIR = SMS_RAW_DIR / "contacts"
IMPORTS_DIR = SMS_RAW_DIR / "imports"

CLEANED_SMS_DIR = CLEANED_DIR / "sms"
CUSTOMER_DIR = CLEANED_DIR / "customer"
CAMPAIGN_DIR = CLEANED_DIR / "campaigns"
QA_DIR = CLEANED_DIR / "qa"
REPLIES_DIR = CLEANED_DIR / "replies"

folders = [
    RAW_DIR,
    CLEANED_DIR,
    EXPORT_DIR,
    SMS_RAW_DIR,
    SC_DIR,
    DR_DIR,
    CONTACTS_DIR,
    IMPORTS_DIR,
    CLEANED_SMS_DIR,
    CUSTOMER_DIR,
    CAMPAIGN_DIR,
    QA_DIR,
    REPLIES_DIR,
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Workbook 16 directory structure verified.")

Workbook 16 directory structure verified.


## 16.3 Source Data Inventory

This section confirms the files available for Workbook 16.

The source data is grouped into three active categories:

- Campaign summary screenshots
- Delivery report CSV files
- SimpleTexting contact exports

This inventory step creates a quick audit trail before analysis begins.

In [59]:
sc_files = sorted([file for file in SC_DIR.glob("*") if not file.name.startswith(".")])
dr_files = sorted(DR_DIR.glob("*.csv"))
contact_files = sorted(CONTACTS_DIR.glob("*.csv"))
import_files = sorted(IMPORTS_DIR.glob("*"))

print("Campaign summary files:", len(sc_files))
for file in sc_files:
    print(" -", file.name)

print("\nDelivery report files:", len(dr_files))
for file in dr_files:
    print(" -", file.name)

print("\nSimpleTexting contact files:", len(contact_files))
for file in contact_files:
    print(" -", file.name)

print("\nImport files:", len(import_files))
for file in import_files:
    print(" -", file.name)

Campaign summary files: 8
 - d1_a_sc.png
 - d1_b_sc.png
 - d2 _sc.png
 - d3_2_sc.png
 - d3_sc.png
 - d4_sc.png
 - d5_sc.png
 - d6_sc.png

Delivery report files: 8
 - d1_a_dr.csv
 - d1_b_dr.csv
 - d2_dr.csv
 - d3_2_dr.csv
 - d3_dr.csv
 - d4_dr.csv
 - d5_dr.csv
 - d6_dr.csv

SimpleTexting contact files: 8
 - d1_a_contacts.csv
 - d1_b_contacts.csv
 - d2_contacts.csv
 - d3_2_contacts.csv
 - d3_contacts.csv
 - d4_contacts.csv
 - d5_contacts.csv
 - d6_contacts.csv

Import files: 0


## 16.4 Campaign Timeline

The campaign was launched in multiple waves because SimpleTexting daily sending limits required the customer list to be split into smaller batches.

The final campaign structure included:

- D1_A
- D1_B
- D2
- D3
- D3_2
- D4
- D5
- D6

The timeline is documented manually because it explains how the campaign evolved and why multiple campaign batches exist.

In [60]:
campaign_timeline = pd.DataFrame([
    {
        "campaign": "D1_A",
        "campaign_group": "Initial Test",
        "description": "First launch batch after Pizza House reopening message was prepared.",
        "notes": "Small batch used to begin campaign delivery.",
    },
    {
        "campaign": "D1_B",
        "campaign_group": "Initial Test",
        "description": "Second launch batch sent after D1_A.",
        "notes": "Continued initial campaign rollout.",
    },
    {
        "campaign": "D2",
        "campaign_group": "Scale Wave",
        "description": "Larger campaign wave after initial batches.",
        "notes": "Daily limits and platform behavior required campaign management.",
    },
    {
        "campaign": "D3",
        "campaign_group": "Restart Wave",
        "description": "Campaign wave affected by campaign restart / overlap behavior.",
        "notes": "Tracked separately to preserve accurate source attribution.",
    },
    {
        "campaign": "D3_2",
        "campaign_group": "System Constraint Wave",
        "description": "Additional D3-related campaign batch created because of platform constraints.",
        "notes": "Kept as its own campaign to avoid mixing source files.",
    },
    {
        "campaign": "D4",
        "campaign_group": "Weekend Wave",
        "description": "Follow-up campaign batch after earlier waves.",
        "notes": "Part of continued customer activation rollout.",
    },
    {
        "campaign": "D5",
        "campaign_group": "Final Wave",
        "description": "Final large customer activation wave.",
        "notes": "Sent after campaign structure was stabilized.",
    },
    {
        "campaign": "D6",
        "campaign_group": "Final Wave",
        "description": "Final remaining customer activation wave.",
        "notes": "Completed remaining customer outreach.",
    },
])

campaign_timeline.to_csv(CLEANED_SMS_DIR / "campaign_timeline.csv", index=False)
print("Exported:", CLEANED_SMS_DIR / "campaign_timeline.csv")
campaign_timeline

Exported: ../data/cleaned/sms/campaign_timeline.csv


,campaign,campaign_group,description,notes
0,D1_A,Initial Test,First launch batch after Pizza House reopening...,Small batch used to begin campaign delivery.
1,D1_B,Initial Test,Second launch batch sent after D1_A.,Continued initial campaign rollout.
2,D2,Scale Wave,Larger campaign wave after initial batches.,Daily limits and platform behavior required ca...
3,D3,Restart Wave,Campaign wave affected by campaign restart / o...,Tracked separately to preserve accurate source...
4,D3_2,System Constraint Wave,Additional D3-related campaign batch created b...,Kept as its own campaign to avoid mixing sourc...
5,D4,Weekend Wave,Follow-up campaign batch after earlier waves.,Part of continued customer activation rollout.
6,D5,Final Wave,Final large customer activation wave.,Sent after campaign structure was stabilized.
7,D6,Final Wave,Final remaining customer activation wave.,Completed remaining customer outreach.


## 16.5 Campaign Summary Table

SimpleTexting campaign summary screenshots provide campaign-level information that is not always available through delivery report exports.

This section creates a structured campaign summary table that can be updated from screenshots.

The table is intentionally manual because screenshot metrics must be verified before they are used in executive reporting.

In [61]:
campaign_summary = pd.DataFrame([
    {"campaign": "D1_A", "campaign_group": "Initial Test", "contacts": 498, "send_date": "2026-06-08", "send_time": "15:30", "total_sent": np.nan, "delivered": np.nan, "failed": np.nan, "success_rate": np.nan, "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."},
    {"campaign": "D1_B", "campaign_group": "Initial Test", "contacts": 493, "send_date": "2026-06-08", "send_time": "15:45", "total_sent": np.nan, "delivered": np.nan, "failed": np.nan, "success_rate": np.nan, "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."},
    {"campaign": "D2", "campaign_group": "Scale Wave", "contacts": 2468, "send_date": "2026-06-09", "send_time": "various", "total_sent": np.nan, "delivered": np.nan, "failed": np.nan, "success_rate": np.nan, "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."},
    {"campaign": "D3", "campaign_group": "Restart Wave", "contacts": 996, "send_date": "2026-06-12", "send_time": "15:30", "total_sent": np.nan, "delivered": np.nan, "failed": np.nan, "success_rate": np.nan, "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."},
    {"campaign": "D3_2", "campaign_group": "System Constraint Wave", "contacts": np.nan, "send_date": "2026-06-10", "send_time": "various", "total_sent": np.nan, "delivered": np.nan, "failed": np.nan, "success_rate": np.nan, "notes": "Update contact and delivery metrics from screenshots."},
    {"campaign": "D4", "campaign_group": "Weekend Wave", "contacts": 1501, "send_date": "2026-06-14", "send_time": "14:00", "total_sent": np.nan, "delivered": np.nan, "failed": np.nan, "success_rate": np.nan, "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."},
    {"campaign": "D5", "campaign_group": "Final Wave", "contacts": 1509, "send_date": "2026-06-16", "send_time": "16:00", "total_sent": np.nan, "delivered": np.nan, "failed": np.nan, "success_rate": np.nan, "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."},
    {"campaign": "D6", "campaign_group": "Final Wave", "contacts": 1511, "send_date": "2026-06-17", "send_time": "16:00", "total_sent": np.nan, "delivered": np.nan, "failed": np.nan, "success_rate": np.nan, "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."},
])

campaign_summary["send_date"] = pd.to_datetime(campaign_summary["send_date"])

campaign_summary.to_csv(CLEANED_SMS_DIR / "campaign_summary.csv", index=False)
campaign_summary.to_csv(EXPORT_DIR / "campaign_summary.csv", index=False)

print("Exported:", CLEANED_SMS_DIR / "campaign_summary.csv")
print("Exported:", EXPORT_DIR / "campaign_summary.csv")
campaign_summary

Exported: ../data/cleaned/sms/campaign_summary.csv
Exported: ../data/exports/campaign_summary.csv


,campaign,campaign_group,contacts,send_date,send_time,total_sent,delivered,failed,success_rate,notes
0,D1_A,Initial Test,498.0,2026-06-08,15:30,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
1,D1_B,Initial Test,493.0,2026-06-08,15:45,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
2,D2,Scale Wave,2468.0,2026-06-09,various,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
3,D3,Restart Wave,996.0,2026-06-12,15:30,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
4,D3_2,System Constraint Wave,NaN,2026-06-10,various,NaN,NaN,NaN,NaN,Update contact and delivery metrics from scree...
5,D4,Weekend Wave,1501.0,2026-06-14,14:00,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
6,D5,Final Wave,1509.0,2026-06-16,16:00,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
7,D6,Final Wave,1511.0,2026-06-17,16:00,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...


## 16.6 Delivery Report Pipeline

Delivery reports are the structured SMS delivery source data exported from SimpleTexting.

This section imports each delivery report, standardizes column names, attaches the campaign name from the file name, and appends all reports into one delivery master table.

In [62]:
delivery_frames = []

for path in sorted(DR_DIR.glob("*_dr.csv")):
    campaign = path.stem.replace("_dr", "").upper()
    temp = pd.read_csv(path)

    temp.columns = (
        temp.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

    temp["campaign"] = campaign
    temp["source_file"] = path.name
    delivery_frames.append(temp)

    print(f"Loaded {campaign}: {len(temp):,} rows")

if delivery_frames:
    delivery_master = pd.concat(delivery_frames, ignore_index=True)
else:
    delivery_master = pd.DataFrame()
    print("No delivery report files found.")

print("Delivery master records:", len(delivery_master))
delivery_master.head()

Loaded D1_A: 497 rows
Loaded D1_B: 493 rows
Loaded D2: 2,975 rows
Loaded D3_2: 1,930 rows
Loaded D3: 866 rows
Loaded D4: 1,419 rows
Loaded D5: 1,391 rows
Loaded D6: 1,348 rows
Delivery master records: 10919


,phone_number,first_name,last_name,delivery_status,campaign,source_file
0,5304094821,22242,4220 Stockton Blvd,Opted Out,D1_A,d1_a_dr.csv
1,9164902587,12882,6 Lopis Crt,Delivered,D1_A,d1_a_dr.csv
2,9168852020,7970,5622 Florin Rd #70,Undelivered - Hard Bounce,D1_A,d1_a_dr.csv
3,9164902538,756,3610 25 Ave,Undelivered - Hard Bounce,D1_A,d1_a_dr.csv
4,2065811608,14226,4100 49th Ave Building 2 #15,Delivered,D1_A,d1_a_dr.csv


## 16.7 Delivery Data Cleaning

This section standardizes phone numbers and delivery statuses.

The cleaned fields make it possible to summarize delivery performance, identify invalid records, find duplicate contacts, and later compare delivery records against SimpleTexting contact export records.

In [63]:
if not delivery_master.empty:
    phone_candidates = [col for col in delivery_master.columns if "phone" in col]
    phone_col = phone_candidates[0] if phone_candidates else None

    if phone_col:
        delivery_master["phone_clean"] = (
            delivery_master[phone_col]
            .astype(str)
            .str.replace(r"\D", "", regex=True)
        )
    else:
        delivery_master["phone_clean"] = np.nan

    status_candidates = [col for col in delivery_master.columns if "status" in col]
    status_col = status_candidates[0] if status_candidates else None

    if status_col:
        delivery_master["delivery_status_clean"] = (
            delivery_master[status_col]
            .astype(str)
            .str.strip()
            .str.lower()
        )
    else:
        delivery_master["delivery_status_clean"] = np.nan

    delivery_master.to_csv(CLEANED_SMS_DIR / "delivery_report_master.csv", index=False)

    print("Exported:", CLEANED_SMS_DIR / "delivery_report_master.csv")
    print("Phone column used:", phone_col)
    print("Status column used:", status_col)
else:
    print("Delivery master is empty. Add delivery report CSV files before running this section.")

delivery_master.head()

Exported: ../data/cleaned/sms/delivery_report_master.csv
Phone column used: phone_number
Status column used: delivery_status


,phone_number,first_name,last_name,delivery_status,campaign,source_file,phone_clean,delivery_status_clean
0,5304094821,22242,4220 Stockton Blvd,Opted Out,D1_A,d1_a_dr.csv,5304094821,opted out
1,9164902587,12882,6 Lopis Crt,Delivered,D1_A,d1_a_dr.csv,9164902587,delivered
2,9168852020,7970,5622 Florin Rd #70,Undelivered - Hard Bounce,D1_A,d1_a_dr.csv,9168852020,undelivered - hard bounce
3,9164902538,756,3610 25 Ave,Undelivered - Hard Bounce,D1_A,d1_a_dr.csv,9164902538,undelivered - hard bounce
4,2065811608,14226,4100 49th Ave Building 2 #15,Delivered,D1_A,d1_a_dr.csv,2065811608,delivered


## 16.8 Delivery Status Summary

Delivery status provides the first measurement of SMS campaign execution.

This section summarizes delivery outcomes by campaign so each batch can be evaluated before customer replies are analyzed.

In [64]:
if not delivery_master.empty:
    delivery_status_summary = (
        delivery_master
        .groupby(["campaign", "delivery_status_clean"], dropna=False)
        .size()
        .reset_index(name="record_count")
        .sort_values(["campaign", "record_count"], ascending=[True, False])
    )

    delivery_status_summary.to_csv(
        CLEANED_SMS_DIR / "delivery_status_summary.csv",
        index=False
    )

    print("Exported:", CLEANED_SMS_DIR / "delivery_status_summary.csv")
else:
    delivery_status_summary = pd.DataFrame(
        columns=["campaign", "delivery_status_clean", "record_count"]
    )
    print("Delivery status summary not created because delivery master is empty.")

delivery_status_summary

Exported: ../data/cleaned/sms/delivery_status_summary.csv


,campaign,delivery_status_clean,record_count
0,D1_A,delivered,323
2,D1_A,undelivered - hard bounce,92
3,D1_A,undelivered - soft bounce,43
1,D1_A,opted out,39
4,D1_B,delivered,342
6,D1_B,undelivered - hard bounce,95
5,D1_B,opted out,29
7,D1_B,undelivered - soft bounce,27
8,D2,delivered,2005
10,D2,undelivered - hard bounce,546


## 16.9 Data Quality Outputs

Delivery reports can be used to identify invalid numbers and duplicate phone records.

These outputs are operationally valuable because they improve the quality of future marketing campaigns and reduce wasted sends.

In [65]:
if not delivery_master.empty:
    invalid_keywords = ["invalid", "failed", "undelivered", "error"]

    invalid_numbers = delivery_master[
        delivery_master["delivery_status_clean"]
        .astype(str)
        .str.contains("|".join(invalid_keywords), na=False)
    ].copy()

    duplicate_phones = (
        delivery_master[delivery_master["phone_clean"].notna()]
        .groupby("phone_clean")
        .size()
        .reset_index(name="record_count")
        .query("record_count > 1")
        .sort_values("record_count", ascending=False)
    )

    invalid_numbers.to_csv(CLEANED_SMS_DIR / "invalid_numbers.csv", index=False)
    duplicate_phones.to_csv(CLEANED_SMS_DIR / "duplicate_phones.csv", index=False)

    print("Exported:", CLEANED_SMS_DIR / "invalid_numbers.csv")
    print("Exported:", CLEANED_SMS_DIR / "duplicate_phones.csv")
    print("Invalid number records:", len(invalid_numbers))
    print("Duplicate phone records:", len(duplicate_phones))
else:
    invalid_numbers = pd.DataFrame()
    duplicate_phones = pd.DataFrame()
    print("Data quality outputs not created because delivery master is empty.")

Exported: ../data/cleaned/sms/invalid_numbers.csv
Exported: ../data/cleaned/sms/duplicate_phones.csv
Invalid number records: 2561
Duplicate phone records: 1524


## 16.10 Initial Activation Funnel

This section creates the first version of the SMS activation funnel using available delivery data.

Verified response metrics are added after the SimpleTexting contact export is parsed and classified.

In [66]:
if not delivery_master.empty:
    total_delivery_records = len(delivery_master)
    unique_phone_records = delivery_master["phone_clean"].nunique()
    invalid_record_count = len(invalid_numbers)

    activation_funnel = pd.DataFrame([
        {"stage": "Delivery Report Records", "count": total_delivery_records},
        {"stage": "Unique Phone Numbers", "count": unique_phone_records},
        {"stage": "Invalid / Failed Records", "count": invalid_record_count},
    ])
else:
    activation_funnel = pd.DataFrame([
        {"stage": "Delivery Report Records", "count": 0},
        {"stage": "Unique Phone Numbers", "count": 0},
        {"stage": "Invalid / Failed Records", "count": 0},
    ])

activation_funnel.to_csv(
    EXPORT_DIR / "sms_activation_funnel_initial.csv",
    index=False
)

print("Exported:", EXPORT_DIR / "sms_activation_funnel_initial.csv")
activation_funnel

Exported: ../data/exports/sms_activation_funnel_initial.csv


,stage,count
0,Delivery Report Records,10919
1,Unique Phone Numbers,9377
2,Invalid / Failed Records,2561


## 16.11 Import SimpleTexting Contacts

SimpleTexting contact exports provide the primary data source for customer reply analysis.

Each contact record includes the customer's phone number, imported contact information, opt-in status, unsubscribe status, and complete SMS conversation history.

These contact exports serve as the foundation for reply classification, customer matching, and SMS engagement analytics throughout the remainder of this workbook.

In [67]:
contacts_raw = pd.read_csv(
    CONTACTS_DIR / "d1_a_contacts.csv"
)

print(f"Contacts imported: {len(contacts_raw):,}")
display(contacts_raw.head())

Contacts imported: 498


,Number,First Name,Last Name,Email,Birthday,Note,Create Date,Opt-in method,Unsubscribed,Text,Inbox
0,2065811608,14226,4100 49th Ave Building 2 #15,NaN,NaN,NaN,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),NaN,NaN,06/08/2026 03:31 PM > Pizza House has moved!No...
1,2092088773,16600,2050 53rd Ave,NaN,NaN,NaN,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),NaN,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...
2,2092848111,1666,5804 63 St,NaN,NaN,NaN,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),NaN,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...
3,2093902376,4194,2487 67 Ave,NaN,NaN,NaN,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),NaN,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...
4,2095619339,7959,4100 49 Ave #62,NaN,NaN,NaN,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),NaN,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...


## 16.12 Clean SimpleTexting Contacts

The SimpleTexting contacts export contains customer phone numbers, imported address fields, opt-in metadata, unsubscribe status, outbound campaign text, and full conversation history.

This section standardizes the export into a clean contacts table by normalizing column names, cleaning phone numbers, rebuilding the imported address field, and preserving the Inbox thread for reply parsing.

In [68]:
contacts_clean = contacts_raw.copy()

contacts_clean.columns = (
    contacts_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

contacts_clean["phone_clean"] = (
    contacts_clean["number"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
)

contacts_clean["address_raw"] = (
    contacts_clean["first_name"].fillna("").astype(str).str.strip()
    + " "
    + contacts_clean["last_name"].fillna("").astype(str).str.strip()
).str.strip()

contacts_clean["inbox"] = (
    contacts_clean["inbox"]
    .fillna("")
    .astype(str)
    .str.strip()
)

contacts_clean["has_inbox_thread"] = contacts_clean["inbox"].ne("")

contacts_clean["unsubscribed_flag"] = (
    contacts_clean["unsubscribed"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

contacts_clean = contacts_clean[
    [
        "phone_clean",
        "address_raw",
        "create_date",
        "opt_in_method",
        "unsubscribed_flag",
        "text",
        "inbox",
        "has_inbox_thread",
    ]
].copy()

contacts_clean.to_csv(
    CLEANED_SMS_DIR / "simpletexting_contacts_clean.csv",
    index=False
)

print(f"Contacts cleaned: {len(contacts_clean):,}")
print(f"Contacts with inbox thread: {contacts_clean['has_inbox_thread'].sum():,}")
print("Exported:", CLEANED_SMS_DIR / "simpletexting_contacts_clean.csv")

display(contacts_clean.head(20))

Contacts cleaned: 498
Contacts with inbox thread: 497
Exported: ../data/cleaned/sms/simpletexting_contacts_clean.csv


,phone_clean,address_raw,create_date,opt_in_method,unsubscribed_flag,text,inbox,has_inbox_thread
0,2065811608,14226 4100 49th Ave Building 2 #15,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:31 PM > Pizza House has moved!No...,True
1,2092088773,16600 2050 53rd Ave,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...,True
2,2092848111,1666 5804 63 St,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...,True
3,2093902376,4194 2487 67 Ave,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...,True
4,2095619339,7959 4100 49 Ave #62,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...,True
5,2096632174,19948 6439 Rancho Adobe Dr,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:31 PM > Pizza House has moved!No...,True
6,2096651647,7452 8200 Elder Creek Rd,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:33 PM > Pizza House has moved!No...,True
7,2097522780,23916 5825 61st,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:32 PM > Pizza House has moved!No...,True
8,2098098402,20875 6060 40 Ave,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:38 PM > Pizza House has moved!No...,True
9,2164509592,14536 3644 18 Ave,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:31 PM > Pizza House has moved!No...,True


## 16.13 Parse Inbox Conversations

The Inbox field contains the complete SMS conversation history for each contact.

This section separates outbound Pizza House messages from inbound customer replies by identifying the message direction contained within each conversation thread. Parsed conversation fields are used throughout the remainder of the workbook for customer engagement analysis.

In [69]:
thread_pattern = (
    r"(\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}\s+[AP]M\s+[><])"
)

parsed_threads = []

for _, row in contacts_clean.iterrows():
    outbound_messages = []
    inbound_replies = []
    inbox = str(row["inbox"])

    if inbox.strip():
        thread = re.split(thread_pattern, inbox)

        for i in range(1, len(thread), 2):
            direction = thread[i]
            message = thread[i + 1].strip()

            if direction.endswith(">"):
                outbound_messages.append(message)
            elif direction.endswith("<"):
                inbound_replies.append(message)

    parsed_threads.append({
        "phone_clean": row["phone_clean"],
        "address_raw": row["address_raw"],
        "outbound_message_count": len(outbound_messages),
        "customer_reply_count": len(inbound_replies),
        "customer_replied_flag": len(inbound_replies) > 0,
        "first_customer_reply": inbound_replies[0] if inbound_replies else "",
        "all_customer_replies": " | ".join(inbound_replies),
        "all_outbound_messages": " | ".join(outbound_messages),
    })

contacts_threads = pd.DataFrame(parsed_threads)

contacts_replies = contacts_clean.merge(
    contacts_threads,
    on=["phone_clean", "address_raw"],
    how="left"
)

contacts_replies.to_csv(
    CLEANED_SMS_DIR / "simpletexting_contacts_replies.csv",
    index=False
)

print(f"Contacts parsed: {len(contacts_replies):,}")
print(f"Customer replies: {contacts_replies['customer_replied_flag'].sum():,}")
print("Exported:", CLEANED_SMS_DIR / "simpletexting_contacts_replies.csv")

display(
    contacts_replies[
        [
            "phone_clean",
            "address_raw",
            "customer_reply_count",
            "first_customer_reply",
            "all_customer_replies",
        ]
    ].head(20)
)

Contacts parsed: 498
Customer replies: 65
Exported: ../data/cleaned/sms/simpletexting_contacts_replies.csv


,phone_clean,address_raw,customer_reply_count,first_customer_reply,all_customer_replies
0,2065811608,14226 4100 49th Ave Building 2 #15,0,,
1,2092088773,16600 2050 53rd Ave,0,,
2,2092848111,1666 5804 63 St,0,,
3,2093902376,4194 2487 67 Ave,0,,
4,2095619339,7959 4100 49 Ave #62,0,,
5,2096632174,19948 6439 Rancho Adobe Dr,0,,
6,2096651647,7452 8200 Elder Creek Rd,0,,
7,2097522780,23916 5825 61st,0,,
8,2098098402,20875 6060 40 Ave,0,,
9,2164509592,14536 3644 18 Ave,0,,


## 16.14 Reply Classification

Customer replies are classified using the Workbook 16 verified response framework.

Because some coupon responses were delayed, positive replies, HELP messages, short confirmation signals, and positive customer engagement are treated as verified engagement when the customer intent is clearly positive.

Coupon fulfillment is identified from outbound messages that include the Pizza House coupon code.

In [70]:
verified_terms = [
    "yes",
    "y",
    "yep",
    "yeah",
    "ok",
    "okay",
    "si",
    "sí",
    "p",
    "👍",
    "❤️",
    "😊",
    "😁",
]

question_terms = [
    "where",
    "address",
    "location",
    "cross",
    "hours",
    "open",
    "when",
    "what time",
]

wrong_number_terms = [
    "wrong",
    "not me",
    "remove",
]

negative_terms = [
    "no",
    "nah",
    "nope",
    "don't",
    "dont",
]

def classify_customer_reply(reply_text):
    value = str(reply_text).strip().lower()

    if value == "":
        return "NO_REPLY"
    if "stop" in value:
        return "STOP"
    if any(term in value for term in wrong_number_terms):
        return "WRONG_NUMBER"
    if "help" in value:
        return "VERIFIED_RESPONSE"
    if (
        value in verified_terms
        or value.startswith("yes")
        or any(term in value for term in verified_terms)
    ):
        return "VERIFIED_RESPONSE"
    if any(term in value for term in question_terms):
        return "QUESTION"
    if any(term in value for term in negative_terms):
        return "NEGATIVE"
    return "OTHER"

contacts_replies["reply_category"] = (
    contacts_replies["all_customer_replies"]
    .apply(classify_customer_reply)
)

contacts_replies["verified_response_flag"] = (
    contacts_replies["reply_category"]
    .eq("VERIFIED_RESPONSE")
)

contacts_replies["coupon_sent_flag"] = (
    contacts_replies["all_outbound_messages"]
    .str.lower()
    .str.contains(
        "thanks for supporting pizza house|pizzahouse10|10% off code",
        na=False,
        regex=True
    )
)

contacts_replies["follow_up_needed"] = (
    contacts_replies["customer_replied_flag"]
    & ~contacts_replies["coupon_sent_flag"]
    & contacts_replies["reply_category"].isin(["VERIFIED_RESPONSE", "QUESTION"])
)

reply_classification_summary = (
    contacts_replies
    .groupby("reply_category")
    .size()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

contacts_replies.to_csv(
    CLEANED_SMS_DIR / "simpletexting_contacts_replies_classified.csv",
    index=False
)

print(f"Contacts classified: {len(contacts_replies):,}")
print(f"Verified responses: {contacts_replies['verified_response_flag'].sum():,}")
print(f"Coupons sent: {contacts_replies['coupon_sent_flag'].sum():,}")
print(f"Follow-up needed: {contacts_replies['follow_up_needed'].sum():,}")
print("Exported:", CLEANED_SMS_DIR / "simpletexting_contacts_replies_classified.csv")

display(reply_classification_summary)

Contacts classified: 498
Verified responses: 26
Coupons sent: 26
Follow-up needed: 0
Exported: ../data/cleaned/sms/simpletexting_contacts_replies_classified.csv


,reply_category,record_count
0,NO_REPLY,433
1,STOP,39
2,VERIFIED_RESPONSE,26


## 16.15 Parse Contact Addresses

The SimpleTexting contacts export stores the imported customer address in the contact name fields.

The first numeric value often represents a SimpleTexting contact identifier rather than the customer street number. This section removes that identifier, standardizes customer addresses, and creates an `address_join_key` used for data quality validation after phone-based customer matching.

Phone number remains the canonical integration key. Address is used as a secondary QA field.

In [71]:
contacts_address = contacts_replies.copy()

parsed_columns = [
    "address_clean",
    "street_number",
    "street_name",
    "street_suffix",
    "unit",
    "address_join_key",
]

contacts_address = contacts_address.drop(
    columns=[c for c in parsed_columns if c in contacts_address.columns],
    errors="ignore"
)

STREET_SUFFIXES = {
    "ST", "AVE", "DR", "RD",
    "BLVD", "CT", "CIR",
    "WAY", "LN", "PL",
    "PKWY", "TER", "TRL", "HWY",
}

UNIT_PREFIXES = (
    "APT",
    "UNIT",
    "#",
    "ROOM",
    "RM",
    "STE",
    "SUITE",
    "BUILDING",
)

ocr_corrections = {
    "BIV": "BLVD",
    "STOCKTEN": "STOCKTON",
    "PK": "PKWY",
    "OR": "DR",
    "AVENUE": "AVE",
    "STREET": "ST",
    "ROAD": "RD",
    "DRIVE": "DR",
    "BOULEVARD": "BLVD",
    "COURT": "CT",
    "CIRCLE": "CIR",
    "PARKWAY": "PKWY",
    "LANE": "LN",
    "PLACE": "PL",
    "TERRACE": "TER",
    "TRAIL": "TRL",
    "HIGHWAY": "HWY",
}

ordinal_streets = {
    "10": "10TH",
    "18": "18TH",
    "23": "23RD",
    "25": "25TH",
    "33": "33RD",
    "35": "35TH",
    "37": "37TH",
    "39": "39TH",
    "40": "40TH",
    "42": "42ND",
    "46": "46TH",
    "47": "47TH",
    "48": "48TH",
    "49": "49TH",
    "50": "50TH",
    "51": "51ST",
    "54": "54TH",
    "55": "55TH",
    "56": "56TH",
    "61": "61ST",
    "63": "63RD",
    "66": "66TH",
    "67": "67TH",
    "69": "69TH",
}

parsed_rows = []

for address in contacts_address["address_raw"]:
    address_clean = None
    street_number = None
    street_name = None
    street_suffix = None
    unit = None
    address_join_key = None

    if pd.notna(address):
        text = str(address).upper().strip()
        text = re.sub(r"\b\d+D\d*\b", "", text)
        text = re.sub(r"@\d+", "", text)
        text = text.replace("&", " ")
        text = re.sub(r"\s+", " ", text).strip()

        for bad, good in ocr_corrections.items():
            text = re.sub(rf"\b{bad}\b", good, text)

        text = re.sub(
            r"(ST|AVE|DR|RD|BLVD|CT|CIR|WAY|LN|PL|PKWY)(APT|UNIT|ROOM|RM|STE|SUITE|BUILDING)",
            r"\1 \2",
            text
        )

        tokens = text.split()

        if (
            len(tokens) >= 3
            and tokens[0].isdigit()
            and tokens[1].isdigit()
            and tokens[2] not in STREET_SUFFIXES
        ):
            tokens = tokens[1:]

        address_clean = " ".join(tokens)

        if tokens and tokens[0].isdigit():
            street_number = tokens.pop(0)

        for i, token in enumerate(tokens):
            token_upper = token.upper()
            if (
                token_upper.startswith(UNIT_PREFIXES)
                or "#" in token_upper
            ):
                unit = " ".join(tokens[i:])
                tokens = tokens[:i]
                break

        for i, token in enumerate(tokens):
            token_upper = token.upper()
            if token_upper in STREET_SUFFIXES:
                street_suffix = token_upper
                tokens = tokens[:i]
                break

        street_name = " ".join(tokens)
        street_name = street_name.replace("&", "").strip()
        street_name = re.sub(r"\s+", " ", street_name).strip()

        if street_name in ordinal_streets:
            street_name = ordinal_streets[street_name]

        join_parts = [street_number]

        if street_name:
            join_parts.append(street_name.replace(" ", "_"))

        if street_suffix:
            join_parts.append(street_suffix)

        address_join_key = "_".join([part for part in join_parts if part])

    parsed_rows.append({
        "address_clean": address_clean,
        "street_number": street_number,
        "street_name": street_name,
        "street_suffix": street_suffix,
        "unit": unit,
        "address_join_key": address_join_key,
    })

parsed_rows = pd.DataFrame(parsed_rows)

contacts_address = pd.concat(
    [contacts_address, parsed_rows],
    axis=1
)

contacts_address.to_csv(
    CLEANED_SMS_DIR / "simpletexting_contacts_address_parsed.csv",
    index=False
)

parsed_addresses = contacts_address["address_join_key"].notna().sum()
total_contacts = len(contacts_address)
parse_rate = parsed_addresses / total_contacts

print(f"Contacts parsed: {total_contacts:,}")
print(f"Addresses parsed: {parsed_addresses:,}")
print(f"Parse rate: {parse_rate:.1%}")
print("Exported:", CLEANED_SMS_DIR / "simpletexting_contacts_address_parsed.csv")

display(
    contacts_address[
        [
            "phone_clean",
            "address_raw",
            "address_clean",
            "street_number",
            "street_name",
            "street_suffix",
            "unit",
            "address_join_key",
        ]
    ].head(20)
)

Contacts parsed: 498
Addresses parsed: 498
Parse rate: 100.0%
Exported: ../data/cleaned/sms/simpletexting_contacts_address_parsed.csv


,phone_clean,address_raw,address_clean,street_number,street_name,street_suffix,unit,address_join_key
0,2065811608,14226 4100 49th Ave Building 2 #15,4100 49TH AVE BUILDING 2 #15,4100,49TH,AVE,BUILDING 2 #15,4100_49TH_AVE
1,2092088773,16600 2050 53rd Ave,2050 53RD AVE,2050,53RD,AVE,None,2050_53RD_AVE
2,2092848111,1666 5804 63 St,5804 63 ST,5804,63RD,ST,None,5804_63RD_ST
3,2093902376,4194 2487 67 Ave,2487 67 AVE,2487,67TH,AVE,None,2487_67TH_AVE
4,2095619339,7959 4100 49 Ave #62,4100 49 AVE #62,4100,49TH,AVE,#62,4100_49TH_AVE
5,2096632174,19948 6439 Rancho Adobe Dr,6439 RANCHO ADOBE DR,6439,RANCHO ADOBE,DR,None,6439_RANCHO_ADOBE_DR
6,2096651647,7452 8200 Elder Creek Rd,8200 ELDER CREEK RD,8200,ELDER CREEK,RD,None,8200_ELDER_CREEK_RD
7,2097522780,23916 5825 61st,5825 61ST,5825,61ST,None,None,5825_61ST
8,2098098402,20875 6060 40 Ave,6060 40 AVE,6060,40TH,AVE,None,6060_40TH_AVE
9,2164509592,14536 3644 18 Ave,3644 18 AVE,3644,18TH,AVE,None,3644_18TH_AVE


## 16.16 SMS Contact Activation Summary

This section creates an updated activation summary using the SimpleTexting contact export and classified replies.

This summary becomes the new checkpoint before phone-based customer matching.

In [72]:
sms_contact_activation_summary = pd.DataFrame([
    {"metric": "SimpleTexting Contacts", "value": len(contacts_address)},
    {"metric": "Contacts With Inbox Thread", "value": int(contacts_clean["has_inbox_thread"].sum())},
    {"metric": "Customer Replies", "value": int(contacts_replies["customer_replied_flag"].sum())},
    {"metric": "Verified Responses", "value": int(contacts_replies["verified_response_flag"].sum())},
    {"metric": "STOP Replies", "value": int((contacts_replies["reply_category"] == "STOP").sum())},
    {"metric": "Coupons Sent", "value": int(contacts_replies["coupon_sent_flag"].sum())},
    {"metric": "Follow-Up Needed", "value": int(contacts_replies["follow_up_needed"].sum())},
    {"metric": "Addresses Parsed", "value": int(contacts_address["address_join_key"].notna().sum())},
])

sms_contact_activation_summary.to_csv(
    CLEANED_SMS_DIR / "sms_contact_activation_summary.csv",
    index=False
)

sms_contact_activation_summary.to_csv(
    EXPORT_DIR / "sms_contact_activation_summary.csv",
    index=False
)

print("Exported:", CLEANED_SMS_DIR / "sms_contact_activation_summary.csv")
print("Exported:", EXPORT_DIR / "sms_contact_activation_summary.csv")

sms_contact_activation_summary

Exported: ../data/cleaned/sms/sms_contact_activation_summary.csv
Exported: ../data/exports/sms_contact_activation_summary.csv


,metric,value
0,SimpleTexting Contacts,498
1,Contacts With Inbox Thread,497
2,Customer Replies,65
3,Verified Responses,26
4,STOP Replies,39
5,Coupons Sent,26
6,Follow-Up Needed,0
7,Addresses Parsed,498


## 16.17 Customer Activation Integration

This section imports the Pizza House marketing-ready customer dataset developed from the cleaned Workbook 13 customer list.

The marketing-ready customer dataset contains the usable customer records that were later divided into SimpleTexting campaign batches. Workbook 16 integrates SMS engagement results back into this customer list using the normalized `phone_clean` field.

This step begins the customer activation intelligence layer by connecting the original marketing audience to verified responses, opt-outs, invalid numbers, and future campaign eligibility.

Phone number serves as the canonical integration key throughout the remainder of Workbook 16.

In [73]:
CUSTOMER_DIR = CLEANED_DIR / "customer"

customers_clean = pd.read_csv(
    CUSTOMER_DIR / "campaign_sms_test_list.csv"
)

print(f"Marketing-ready customers: {len(customers_clean):,}")

display(customers_clean.head())

Marketing-ready customers: 9,557


,customer_id_clean,Phone Number,Address Line 1
0,1,9162129352,6710 37 Ave
1,4,9162301496,6425 Somis Way
2,5,9163292467,6100 48th Ave #2202
3,6,2792720793,3826 25th Ave
4,7,9162065582,7101 Gerber Rd #216 #8106


## 16.18 Prepare Customer Dataset

This section prepares the marketing-ready customer dataset for integration with the SMS contact-event data.

Customer phone numbers are normalized into a canonical `phone_clean` field by removing non-numeric characters and standardizing U.S. phone numbers to a 10-digit format. A leading country code (`1`) is removed when present.

The original customer address is preserved as `address_raw` for downstream address parsing and QA validation.

In [74]:
customers_prepared = customers_clean.copy()

customers_prepared = customers_prepared.rename(
    columns={
        "Phone Number": "phone_number",
        "Address Line 1": "address_line_1",
    }
)

customers_prepared["phone_clean"] = (
    customers_prepared["phone_number"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
    .str.replace(r"^1(?=\d{10}$)", "", regex=True)
)

customers_prepared["address_raw"] = (
    customers_prepared["address_line_1"]
    .fillna("")
    .str.strip()
)

print(f"Customer records: {len(customers_prepared):,}")

print(
    f"Unique phone numbers: "
    f"{customers_prepared['phone_clean'].nunique():,}"
)

display(
    customers_prepared[
        [
            "customer_id_clean",
            "phone_number",
            "phone_clean",
            "address_raw",
        ]
    ].head(20)
)

Customer records: 9,557
Unique phone numbers: 9,452


,customer_id_clean,phone_number,phone_clean,address_raw
0,1,9162129352,9162129352,6710 37 Ave
1,4,9162301496,9162301496,6425 Somis Way
2,5,9163292467,9163292467,6100 48th Ave #2202
3,6,2792720793,2792720793,3826 25th Ave
4,7,9162065582,9162065582,7101 Gerber Rd #216 #8106
5,8,9167045539,9167045539,5677 23rd St
6,13,9169527365,9169527365,6267 Mlk Jr Blvd Apt 122
7,15,9168064845,9168064845,7809 36th Ave
8,16,9168217817,9168217817,4709 47th St
9,20,2792486208,2792486208,6448 Stockton Blvd Room 27


## 16.19 Parse Customer Addresses

This section applies the Workbook 16 address parsing logic to the Pizza House marketing-ready customer dataset.

The same address standardization rules used for the SimpleTexting contact export are applied to the customer dataset so both sources can be compared using a common `address_join_key`.

Phone number remains the canonical integration key. The parsed customer address fields are used as a secondary QA layer after SMS activation results are integrated back into the customer list.

In [75]:
con = duckdb.connect()

con.register(
    "customers",
    customers_address
)

con.register(
    "contacts",
    contacts_address_all
)

customer_activation_master = con.sql(
    """

    WITH activation_summary AS (

        SELECT

            phone_clean,

            COUNT(*) AS campaign_touch_count,

            STRING_AGG(
                DISTINCT campaign,
                ', '
                ORDER BY campaign
            ) AS campaigns_received,

            SUM(customer_reply_count) AS customer_reply_count,

            BOOL_OR(
                reply_category = 'VERIFIED_RESPONSE'
            ) AS verified_response_any,

            BOOL_OR(
                reply_category = 'STOP'
            ) AS stop_any,

            ANY_VALUE(address_join_key) AS address_join_key_sms

        FROM contacts

        GROUP BY phone_clean

    )

    SELECT

        c.customer_id_clean,

        c.phone_number,

        c.phone_clean,

        c.address_raw,

        c.address_clean AS address_clean_customer,

        c.street_number AS street_number_customer,

        c.street_name AS street_name_customer,

        c.street_suffix AS street_suffix_customer,

        c.unit AS unit_customer,

        c.address_join_key AS address_join_key_customer,

        COALESCE(
            a.campaign_touch_count,
            0
        ) AS campaign_touch_count,

        a.campaigns_received,

        COALESCE(
            a.customer_reply_count,
            0
        ) AS customer_reply_count,

        COALESCE(
            a.verified_response_any,
            FALSE
        ) AS verified_response_any,

        COALESCE(
            a.stop_any,
            FALSE
        ) AS stop_any,

        a.address_join_key_sms,

        CASE

            WHEN a.stop_any THEN 'STOP'

            WHEN a.verified_response_any THEN 'VERIFIED_RESPONSE'

            WHEN a.phone_clean IS NULL THEN 'NOT_CONTACTED'

            ELSE 'NO_RESPONSE'

        END AS activation_status,

        CASE

            WHEN a.stop_any THEN FALSE

            WHEN a.verified_response_any THEN FALSE

            ELSE TRUE

        END AS eligible_for_future_campaign

    FROM customers c

    LEFT JOIN activation_summary a

        ON c.phone_clean = a.phone_clean

    """
).df()

customer_activation_master.to_csv(
    CUSTOMER_DIR / "customer_activation_master.csv",
    index=False
)

print(f"Customer records: {len(customer_activation_master):,}")

print(
    f"Matched SMS contacts: "
    f"{(customer_activation_master['campaign_touch_count'] > 0).sum():,}"
)

print(
    f"Verified responders: "
    f"{customer_activation_master['verified_response_any'].sum():,}"
)

print(
    f"STOP requests: "
    f"{customer_activation_master['stop_any'].sum():,}"
)

print(
    f"Eligible for future campaigns: "
    f"{customer_activation_master['eligible_for_future_campaign'].sum():,}"
)

print(
    "Exported:",
    CUSTOMER_DIR / "customer_activation_master.csv"
)

display(
    customer_activation_master[
        [
            "customer_id_clean",
            "phone_clean",
            "address_raw",
            "address_join_key_customer",
            "campaign_touch_count",
            "campaigns_received",
            "customer_reply_count",
            "activation_status",
            "eligible_for_future_campaign",
        ]
    ].head(20)
)

Customer records: 9,557
Addresses parsed: 9,557
Parse rate: 100.0%


,customer_id_clean,phone_clean,address_raw,address_clean,street_number,street_name,street_suffix,unit,address_join_key
0,1,9162129352,6710 37 Ave,6710 37 AVE,6710,37TH,AVE,None,6710_37TH_AVE
1,4,9162301496,6425 Somis Way,6425 SOMIS WAY,6425,SOMIS,WAY,None,6425_SOMIS_WAY
2,5,9163292467,6100 48th Ave #2202,6100 48TH AVE #2202,6100,48TH,AVE,#2202,6100_48TH_AVE
3,6,2792720793,3826 25th Ave,3826 25TH AVE,3826,25TH,AVE,None,3826_25TH_AVE
4,7,9162065582,7101 Gerber Rd #216 #8106,7101 GERBER RD #216 #8106,7101,GERBER,RD,#216 #8106,7101_GERBER_RD
5,8,9167045539,5677 23rd St,5677 23RD ST,5677,23RD,ST,None,5677_23RD_ST
6,13,9169527365,6267 Mlk Jr Blvd Apt 122,6267 MLK JR BLVD APT 122,6267,MLK JR,BLVD,APT 122,6267_MLK_JR_BLVD
7,15,9168064845,7809 36th Ave,7809 36TH AVE,7809,36TH,AVE,None,7809_36TH_AVE
8,16,9168217817,4709 47th St,4709 47TH ST,4709,47TH,ST,None,4709_47TH_ST
9,20,2792486208,6448 Stockton Blvd Room 27,6448 STOCKTON BLVD ROOM 27,6448,STOCKTON,BLVD,ROOM 27,6448_STOCKTON_BLVD


## 16.20 Import All SimpleTexting Contact Exports

This section imports all SimpleTexting contact export files stored in the raw SMS contacts folder.

Each contact export represents one campaign batch from the Pizza House Grand Reopening SMS campaign. Importing all contact exports together allows Workbook 16 to analyze customer replies across the full campaign instead of only one batch.

The campaign name is extracted from each file name so reply activity can still be traced back to the original campaign source.

In [76]:
contact_files = sorted(CONTACTS_DIR.glob("*.csv"))

contact_frames = []

for path in contact_files:
    campaign = (
        path.stem
        .replace("_contacts", "")
        .replace("_contact", "")
        .upper()
    )
    
    temp = pd.read_csv(path)
    temp["campaign"] = campaign
    temp["source_file"] = path.name
    
    contact_frames.append(temp)
    
    print(f"Loaded {campaign}: {len(temp):,} contacts")

if contact_frames:
    contacts_raw_all = pd.concat(contact_frames, ignore_index=True)
else:
    contacts_raw_all = pd.DataFrame()
    print("No SimpleTexting contact export files found.")

print(f"\nTotal contact records imported: {len(contacts_raw_all):,}")
print(f"Contact export files imported: {len(contact_files):,}")

display(contacts_raw_all.head())

Loaded D1_A: 498 contacts
Loaded D1_B: 493 contacts
Loaded D2: 2,980 contacts
Loaded D3_2: 996 contacts
Loaded D3: 5,483 contacts
Loaded D4: 1,501 contacts
Loaded D5: 1,509 contacts
Loaded D6: 1,511 contacts

Total contact records imported: 14,971
Contact export files imported: 8


,Number,First Name,Last Name,Email,Birthday,Note,Create Date,Opt-in method,Unsubscribed,Text,Inbox,campaign,source_file
0,2065811608,14226,4100 49th Ave Building 2 #15,NaN,NaN,NaN,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),NaN,NaN,06/08/2026 03:31 PM > Pizza House has moved!No...,D1_A,d1_a_contacts.csv
1,2092088773,16600,2050 53rd Ave,NaN,NaN,NaN,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),NaN,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...,D1_A,d1_a_contacts.csv
2,2092848111,1666,5804 63 St,NaN,NaN,NaN,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),NaN,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...,D1_A,d1_a_contacts.csv
3,2093902376,4194,2487 67 Ave,NaN,NaN,NaN,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),NaN,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...,D1_A,d1_a_contacts.csv
4,2095619339,7959,4100 49 Ave #62,NaN,NaN,NaN,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),NaN,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...,D1_A,d1_a_contacts.csv


## 16.21 Clean All SimpleTexting Contact Records

This section standardizes all imported SimpleTexting contact exports into a single contact event dataset.

Each row represents one contact record from one SMS campaign batch. Customer records are intentionally **not** deduplicated at this stage because a customer may appear in multiple campaign batches due to resend activity, campaign segmentation, or platform limitations.

The cleaned contact event dataset preserves campaign history and serves as the foundation for reply parsing, reply classification, customer activation analysis, and future customer-level aggregation.

In [77]:
contacts_clean_all = contacts_raw_all.copy()

contacts_clean_all.columns = (
    contacts_clean_all.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

contacts_clean_all["phone_clean"] = (
    contacts_clean_all["number"]
    .fillna("")
    .astype(str)
    .str.replace(r"\D", "", regex=True)
)

contacts_clean_all["address_raw"] = (
    contacts_clean_all["first_name"]
    .fillna("")
    .astype(str)
    .str.strip()
    + " "
    + contacts_clean_all["last_name"]
    .fillna("")
    .astype(str)
    .str.strip()
).str.strip()

contacts_clean_all["inbox"] = (
    contacts_clean_all["inbox"]
    .fillna("")
    .astype(str)
)

contacts_clean_all["has_inbox_thread"] = (
    contacts_clean_all["inbox"].str.strip() != ""
)

contacts_clean_all["unsubscribed_flag"] = (
    contacts_clean_all["unsubscribed"]
    .notna()
)

contacts_clean_all = contacts_clean_all[
    [
        "campaign",
        "source_file",
        "phone_clean",
        "address_raw",
        "create_date",
        "opt_in_method",
        "unsubscribed_flag",
        "text",
        "inbox",
        "has_inbox_thread",
    ]
].copy()

contacts_clean_all.to_csv(
    CLEANED_SMS_DIR / "simpletexting_contacts_clean.csv",
    index=False
)

print(f"Contact records: {len(contacts_clean_all):,}")
print(f"Unique phone numbers: {contacts_clean_all['phone_clean'].nunique():,}")
print(f"Contacts with inbox thread: {contacts_clean_all['has_inbox_thread'].sum():,}")
print(f"Campaigns represented: {contacts_clean_all['campaign'].nunique()}")

display(contacts_clean_all.head())

Contact records: 14,971
Unique phone numbers: 9,398
Contacts with inbox thread: 14,957
Campaigns represented: 8


,campaign,source_file,phone_clean,address_raw,create_date,opt_in_method,unsubscribed_flag,text,inbox,has_inbox_thread
0,D1_A,d1_a_contacts.csv,2065811608,14226 4100 49th Ave Building 2 #15,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:31 PM > Pizza House has moved!No...,True
1,D1_A,d1_a_contacts.csv,2092088773,16600 2050 53rd Ave,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...,True
2,D1_A,d1_a_contacts.csv,2092848111,1666 5804 63 St,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...,True
3,D1_A,d1_a_contacts.csv,2093902376,4194 2487 67 Ave,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...,True
4,D1_A,d1_a_contacts.csv,2095619339,7959 4100 49 Ave #62,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...,True


## 16.22 Parse All Inbox Conversations

This section parses the Inbox conversation history for every SimpleTexting contact event record.

Each Inbox thread is separated into outbound Pizza House messages and inbound customer replies using timestamp and message direction markers. Because customers may appear in multiple campaign batches, this section preserves contact-event-level reply behavior before the data is later summarized into one customer activation record per phone number.

The parsed conversation fields support reply classification, campaign touch analysis, coupon fulfillment tracking, and customer activation status assignment.

In [78]:
thread_pattern = (
    r"(\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}\s+[AP]M\s+[><])"
)

parsed_threads = []

for _, row in contacts_clean_all.iterrows():
    outbound_messages = []
    inbound_replies = []

    inbox = str(row["inbox"])

    if inbox.strip():
        thread = re.split(thread_pattern, inbox)

        for i in range(1, len(thread), 2):
            direction = thread[i]
            message = thread[i + 1].strip()

            if direction.endswith(">"):
                outbound_messages.append(message)

            elif direction.endswith("<"):
                inbound_replies.append(message)

    parsed_threads.append({
        "campaign": row["campaign"],
        "source_file": row["source_file"],
        "phone_clean": row["phone_clean"],
        "address_raw": row["address_raw"],
        "outbound_message_count": len(outbound_messages),
        "customer_reply_count": len(inbound_replies),
        "customer_replied_flag": len(inbound_replies) > 0,
        "first_customer_reply": inbound_replies[0] if inbound_replies else "",
        "all_customer_replies": " | ".join(inbound_replies),
        "all_outbound_messages": " | ".join(outbound_messages),
    })

contacts_threads_all = pd.DataFrame(parsed_threads)

contacts_replies_all = contacts_clean_all.merge(
    contacts_threads_all,
    on=[
        "campaign",
        "source_file",
        "phone_clean",
        "address_raw",
    ],
    how="left"
)

contacts_replies_all.to_csv(
    CLEANED_SMS_DIR / "simpletexting_contacts_replies.csv",
    index=False
)

print(f"Contact records parsed: {len(contacts_replies_all):,}")
print(f"Unique phone numbers: {contacts_replies_all['phone_clean'].nunique():,}")
print(f"Customer reply contact records: {contacts_replies_all['customer_replied_flag'].sum():,}")
print(f"Unique customers with replies: {contacts_replies_all.loc[contacts_replies_all['customer_replied_flag'], 'phone_clean'].nunique():,}")
print("Exported:", CLEANED_SMS_DIR / "simpletexting_contacts_replies.csv")

display(
    contacts_replies_all[
        [
            "campaign",
            "phone_clean",
            "address_raw",
            "customer_reply_count",
            "first_customer_reply",
            "all_customer_replies",
        ]
    ].head(20)
)

Contact records parsed: 14,971
Unique phone numbers: 9,398
Customer reply contact records: 1,597
Unique customers with replies: 1,010
Exported: ../data/cleaned/sms/simpletexting_contacts_replies.csv


,campaign,phone_clean,address_raw,customer_reply_count,first_customer_reply,all_customer_replies
0,D1_A,2065811608,14226 4100 49th Ave Building 2 #15,0,,
1,D1_A,2092088773,16600 2050 53rd Ave,0,,
2,D1_A,2092848111,1666 5804 63 St,0,,
3,D1_A,2093902376,4194 2487 67 Ave,0,,
4,D1_A,2095619339,7959 4100 49 Ave #62,0,,
5,D1_A,2096632174,19948 6439 Rancho Adobe Dr,0,,
6,D1_A,2096651647,7452 8200 Elder Creek Rd,0,,
7,D1_A,2097522780,23916 5825 61st,0,,
8,D1_A,2098098402,20875 6060 40 Ave,0,,
9,D1_A,2164509592,14536 3644 18 Ave,0,,


## 16.23 Classify Customer Replies

This section classifies each parsed customer reply into standardized engagement categories.

Reply classification transforms raw conversation text into structured customer engagement outcomes that can be summarized across campaigns and later aggregated into the Customer Activation Master.

Because the contact event layer intentionally preserves multiple campaign touches for the same customer, reply classification is performed at the contact-event level before customer-level aggregation.

In [79]:
contacts_classified = contacts_replies_all.copy()

contacts_classified["reply_text"] = (
    contacts_classified["all_customer_replies"]
    .fillna("")
    .astype(str)
    .str.strip()
)

reply_upper = contacts_classified["reply_text"].str.upper()

contacts_classified["reply_category"] = "NO_REPLY"

contacts_classified.loc[
    reply_upper.str.contains(
        r"\bSTOP\b|\bSTOPALL\b|\bUNSUBSCRIBE\b|\bEND\b|\bCANCEL\b",
        regex=True,
        na=False,
    ),
    "reply_category",
] = "STOP"

contacts_classified.loc[
    (contacts_classified["reply_text"] != "")
    & (contacts_classified["reply_category"] != "STOP"),
    "reply_category",
] = "VERIFIED_RESPONSE"

contacts_classified["verified_response_flag"] = (
    contacts_classified["reply_category"] == "VERIFIED_RESPONSE"
)

contacts_classified["stop_flag"] = (
    contacts_classified["reply_category"] == "STOP"
)

contacts_classified.to_csv(
    CLEANED_SMS_DIR / "simpletexting_contacts_replies_classified.csv",
    index=False,
)

classification_summary = (
    contacts_classified["reply_category"]
    .value_counts()
    .rename_axis("reply_category")
    .reset_index(name="contact_records")
)

print(f"Contact records classified: {len(contacts_classified):,}")
print(f"Verified responses: {contacts_classified['verified_response_flag'].sum():,}")
print(f"STOP requests: {contacts_classified['stop_flag'].sum():,}")
print(f"No reply: {(contacts_classified['reply_category'] == 'NO_REPLY').sum():,}")

display(classification_summary)

display(
    contacts_classified[
        [
            "campaign",
            "phone_clean",
            "reply_category",
            "reply_text",
        ]
    ]
    .query("reply_category != 'NO_REPLY'")
    .head(20)
)

Contact records classified: 14,971
Verified responses: 708
STOP requests: 889
No reply: 13,374


,reply_category,contact_records
0,NO_REPLY,13374
1,STOP,889
2,VERIFIED_RESPONSE,708


,campaign,phone_clean,reply_category,reply_text
16,D1_A,2792097712,VERIFIED_RESPONSE,Yes
21,D1_A,2792219688,VERIFIED_RESPONSE,Yes
26,D1_A,2792314772,VERIFIED_RESPONSE,Yes
112,D1_A,9162202201,VERIFIED_RESPONSE,Yes
156,D1_A,9163181500,VERIFIED_RESPONSE,Yes
164,D1_A,9163460667,VERIFIED_RESPONSE,Yes
166,D1_A,9163496118,VERIFIED_RESPONSE,Yes
172,D1_A,9163807317,VERIFIED_RESPONSE,Yes
208,D1_A,9164708680,VERIFIED_RESPONSE,Yes
229,D1_A,9165104669,VERIFIED_RESPONSE,YES


## 16.24 Parse SimpleTexting Contact Addresses

This section applies the Workbook 16 address parsing logic to all cleaned SimpleTexting contact event records.

During development, additional edge cases were identified while processing the complete multi-campaign dataset. The ordinal street conversion logic was refactored into a reusable function, replacing the earlier lookup-table approach used during initial parser development.

Because the contact event layer includes all campaign exports, the parsed address output preserves campaign-level history while also creating a standardized `address_join_key` for QA validation.

Phone number remains the canonical integration key. Parsed addresses are used as a secondary validation field when SMS activation results are integrated back into the Pizza House customer dataset.

In [80]:
def to_ordinal(value):
    value = int(value)

    if 10 <= value % 100 <= 20:
        suffix = "TH"
    else:
        suffix = {
            1: "ST",
            2: "ND",
            3: "RD",
        }.get(value % 10, "TH")

    return f"{value}{suffix}"


contacts_address_all = contacts_classified.copy()

parsed_columns = [
    "address_clean",
    "street_number",
    "street_name",
    "street_suffix",
    "unit",
    "address_join_key",
]

contacts_address_all = contacts_address_all.drop(
    columns=[c for c in parsed_columns if c in contacts_address_all.columns],
    errors="ignore"
)

parsed_rows = []

for address in contacts_address_all["address_raw"]:
    address_clean = None
    street_number = None
    street_name = None
    street_suffix = None
    unit = None
    address_join_key = None

    if pd.notna(address):

        text = str(address).upper().strip()

        text = re.sub(r"\b\d+D\d*\b", "", text)
        text = re.sub(r"@\d+", "", text)
        text = text.replace("&", " ")
        text = re.sub(r"\s+", " ", text).strip()

        for bad, good in ocr_corrections.items():
            text = re.sub(rf"\b{bad}\b", good, text)

        text = re.sub(
            r"(ST|AVE|DR|RD|BLVD|CT|CIR|WAY|LN|PL|PKWY)(APT|UNIT|ROOM|RM|STE|SUITE|BUILDING)",
            r"\1 \2",
            text,
        )

        tokens = text.split()

        if (
            len(tokens) >= 3
            and tokens[0].isdigit()
            and tokens[1].isdigit()
            and tokens[2] not in STREET_SUFFIXES
        ):
            tokens = tokens[1:]

        address_clean = " ".join(tokens)

        if tokens and tokens[0].isdigit():
            street_number = tokens.pop(0)

        for i, token in enumerate(tokens):

            token_upper = token.upper()

            if (
                token_upper.startswith(UNIT_PREFIXES)
                or "#" in token_upper
            ):
                unit = " ".join(tokens[i:])
                tokens = tokens[:i]
                break

        for i, token in enumerate(tokens):

            token_upper = token.upper()

            if token_upper in STREET_SUFFIXES:
                street_suffix = token_upper
                tokens = tokens[:i]
                break

        street_name = " ".join(tokens)
        street_name = street_name.replace("&", "").strip()
        street_name = re.sub(r"\s+", " ", street_name).strip()

        if re.fullmatch(r"\d+", street_name):
            street_name = to_ordinal(street_name)

        join_parts = [street_number]

        if street_name:
            join_parts.append(street_name.replace(" ", "_"))

        if street_suffix:
            join_parts.append(street_suffix)

        address_join_key = "_".join(
            [part for part in join_parts if part]
        )

    parsed_rows.append({
        "address_clean": address_clean,
        "street_number": street_number,
        "street_name": street_name,
        "street_suffix": street_suffix,
        "unit": unit,
        "address_join_key": address_join_key,
    })

parsed_rows = pd.DataFrame(parsed_rows)

contacts_address_all = pd.concat(
    [contacts_address_all, parsed_rows],
    axis=1
)

contacts_address_all.to_csv(
    CLEANED_SMS_DIR / "simpletexting_contacts_address_parsed.csv",
    index=False,
)

parsed_addresses = contacts_address_all["address_join_key"].notna().sum()
total_contact_records = len(contacts_address_all)
parse_rate = parsed_addresses / total_contact_records

print(f"Contact records parsed: {total_contact_records:,}")
print(f"Addresses parsed: {parsed_addresses:,}")
print(f"Parse rate: {parse_rate:.1%}")

print(
    "Exported:",
    CLEANED_SMS_DIR / "simpletexting_contacts_address_parsed.csv"
)

display(
    contacts_address_all[
        [
            "campaign",
            "phone_clean",
            "address_raw",
            "address_clean",
            "street_number",
            "street_name",
            "street_suffix",
            "unit",
            "address_join_key",
        ]
    ].head(20)
)

Contact records parsed: 14,971
Addresses parsed: 14,971
Parse rate: 100.0%
Exported: ../data/cleaned/sms/simpletexting_contacts_address_parsed.csv


,campaign,phone_clean,address_raw,address_clean,street_number,street_name,street_suffix,unit,address_join_key
0,D1_A,2065811608,14226 4100 49th Ave Building 2 #15,4100 49TH AVE BUILDING 2 #15,4100,49TH,AVE,BUILDING 2 #15,4100_49TH_AVE
1,D1_A,2092088773,16600 2050 53rd Ave,2050 53RD AVE,2050,53RD,AVE,None,2050_53RD_AVE
2,D1_A,2092848111,1666 5804 63 St,5804 63 ST,5804,63RD,ST,None,5804_63RD_ST
3,D1_A,2093902376,4194 2487 67 Ave,2487 67 AVE,2487,67TH,AVE,None,2487_67TH_AVE
4,D1_A,2095619339,7959 4100 49 Ave #62,4100 49 AVE #62,4100,49TH,AVE,#62,4100_49TH_AVE
5,D1_A,2096632174,19948 6439 Rancho Adobe Dr,6439 RANCHO ADOBE DR,6439,RANCHO ADOBE,DR,None,6439_RANCHO_ADOBE_DR
6,D1_A,2096651647,7452 8200 Elder Creek Rd,8200 ELDER CREEK RD,8200,ELDER CREEK,RD,None,8200_ELDER_CREEK_RD
7,D1_A,2097522780,23916 5825 61st,5825 61ST,5825,61ST,None,None,5825_61ST
8,D1_A,2098098402,20875 6060 40 Ave,6060 40 AVE,6060,40TH,AVE,None,6060_40TH_AVE
9,D1_A,2164509592,14536 3644 18 Ave,3644 18 AVE,3644,18TH,AVE,None,3644_18TH_AVE


### 16.24.1 Customer Match Readiness QA

Before integrating the customer and SMS datasets, this section validates the primary integration key.

The normalized `phone_clean` field is compared between the marketing-ready customer dataset and the processed SimpleTexting contact-event dataset to determine how many SMS contact records are expected to successfully match customer records.

This QA step provides a baseline match rate prior to building the Customer Activation Master.

In [81]:
customer_phones = set(customers_prepared["phone_clean"])

sms_phones = set(contacts_address_all["phone_clean"])

matched_phones = customer_phones.intersection(sms_phones)

print(f"Customer phones: {len(customer_phones):,}")
print(f"SMS phones: {len(sms_phones):,}")
print(f"Matching phones: {len(matched_phones):,}")

match_rate = len(matched_phones) / len(customer_phones)

print(f"Phone match rate: {match_rate:.1%}")

Customer phones: 9,452
SMS phones: 9,398
Matching phones: 9,383
Phone match rate: 99.3%


In [82]:
customer_only_phones = customer_phones - sms_phones
sms_only_phones = sms_phones - customer_phones

customer_only_check = customers_prepared[
    customers_prepared["phone_clean"].isin(customer_only_phones)
].copy()

sms_only_check = contacts_address_all[
    contacts_address_all["phone_clean"].isin(sms_only_phones)
].copy()

print(f"Customer-only phones: {len(customer_only_phones):,}")
print(f"SMS-only phones: {len(sms_only_phones):,}")

display(customer_only_check.head())
display(
    sms_only_check[
        [
            "campaign",
            "source_file",
            "phone_clean",
            "address_raw",
        ]
    ].drop_duplicates().head()
)

Customer-only phones: 69
SMS-only phones: 15


,customer_id_clean,phone_number,address_line_1,phone_clean,address_raw
186,581,9166700464,7407 Lemon Hill Ave,9166700464,7407 Lemon Hill Ave
500,1477,27927,5730 Nina Way #10 Upstairs,27927,5730 Nina Way #10 Upstairs
531,1581,9164773175,7235 Elder Creek Rd,9164773175,7235 Elder Creek Rd
703,2083,9164651931,5825 42nd St,9164651931,5825 42nd St
1204,3445,916495368,4525 11 Ave,916495368,4525 11 Ave


,campaign,source_file,phone_clean,address_raw
507,D1_B,d1_b_contacts.csv,2137695517,16837 6448 Stocktan Blvd #26
803,D1_B,d1_b_contacts.csv,9166135393,12609 6311 Samson Blvd #7
1517,D2,d2_contacts.csv,5806444985,8848 5441 80th St
2443,D2,d2_contacts.csv,9164955784,13194 4604 Roosevelt Ave
3079,D2,d2_contacts.csv,9166952327,21356 2176 Perkins Way


In [83]:
print("Customer-only phone length")
display(
    customer_only_check["phone_clean"]
    .astype(str)
    .str.len()
    .value_counts()
)

print("SMS-only phone length")
display(
    sms_only_check["phone_clean"]
    .astype(str)
    .str.len()
    .value_counts()
)

Customer-only phone length


phone_clean
10    18
7     14
9     11
21     9
20     5
11     4
12     3
3      2
5      1
8      1
19     1
Name: count, dtype: int64

SMS-only phone length


phone_clean
10    23
Name: count, dtype: int64

In [84]:
customer_only_check[
    customer_only_check["phone_clean"]
    .astype(str)
    .str.len() == 11
][
    [
        "phone_number",
        "phone_clean",
    ]
].head(20)

,phone_number,phone_clean
1997,27997997035,27997997035
2381,27994657252,27994657252
3980,27998953173,27998953173
8499,27992289721,27992289721


## 16.25 Build Customer Activation Master with SQL

This section integrates the marketing-ready customer dataset with the processed SimpleTexting contact-event dataset using SQL.

Customer phone number serves as the primary integration key across both systems. Because customers may appear in multiple SMS campaign exports, SQL first summarizes campaign activity into a customer-level activation profile before joining the results back to the customer master list.

The Customer Activation Master also preserves customer address fields so verified responders can later be analyzed for flyer distribution, USPS route planning, and neighborhood-level campaign pulse.

This hybrid Python + SQL workflow reflects common business intelligence practices, where Python prepares datasets and SQL performs relational modeling and business logic.

The resulting Customer Activation Master becomes the primary dataset for future SMS campaigns, flyer targeting, suppression lists, engagement reporting, Tableau dashboards, and customer lifecycle analysis.

In [90]:
con = duckdb.connect()

con.register(
    "customers",
    customers_address
)

con.register(
    "contacts",
    contacts_address_all
)

customer_activation_master = con.sql(
    """

    WITH activation_summary AS (

        SELECT

            phone_clean,

            COUNT(*) AS campaign_touch_count,

            MIN(campaign) AS first_campaign_seen,

            MAX(campaign) AS last_campaign_seen,

            SUM(customer_reply_count) AS customer_reply_count,

            BOOL_OR(
                reply_category = 'VERIFIED_RESPONSE'
            ) AS verified_response_any,

            BOOL_OR(
                reply_category = 'STOP'
            ) AS stop_any,

            ANY_VALUE(address_join_key) AS address_join_key_sms

        FROM contacts

        GROUP BY phone_clean

    )

    SELECT

        c.customer_id_clean,

        c.phone_number,

        c.phone_clean,

        c.address_raw,

        c.address_clean AS address_clean_customer,

        c.street_number AS street_number_customer,

        c.street_name AS street_name_customer,

        c.street_suffix AS street_suffix_customer,

        c.unit AS unit_customer,

        c.address_join_key AS address_join_key_customer,

        COALESCE(
            a.campaign_touch_count,
            0
        ) AS campaign_touch_count,

        a.first_campaign_seen,

        a.last_campaign_seen,

        COALESCE(
            a.customer_reply_count,
            0
        ) AS customer_reply_count,

        COALESCE(
            a.verified_response_any,
            FALSE
        ) AS verified_response_any,

        COALESCE(
            a.stop_any,
            FALSE
        ) AS stop_any,

        a.address_join_key_sms,

        CASE

            WHEN a.stop_any THEN 'STOP'

            WHEN a.verified_response_any THEN 'VERIFIED_RESPONSE'

            WHEN a.phone_clean IS NULL THEN 'NOT_CONTACTED'

            ELSE 'NO_RESPONSE'

        END AS activation_status,

        CASE

            WHEN a.stop_any THEN FALSE

            WHEN a.verified_response_any THEN FALSE

            ELSE TRUE

        END AS eligible_for_future_campaign

    FROM customers c

    LEFT JOIN activation_summary a

        ON c.phone_clean = a.phone_clean

    """
).df()

customer_activation_master.to_csv(
    CUSTOMER_DIR / "customer_activation_master.csv",
    index=False
)

print(f"Customer records: {len(customer_activation_master):,}")

print(
    f"Matched SMS contacts: "
    f"{(customer_activation_master['campaign_touch_count'] > 0).sum():,}"
)

print(
    f"Verified responders: "
    f"{customer_activation_master['verified_response_any'].sum():,}"
)

print(
    f"STOP requests: "
    f"{customer_activation_master['stop_any'].sum():,}"
)

print(
    f"Eligible for future campaigns: "
    f"{customer_activation_master['eligible_for_future_campaign'].sum():,}"
)

print(
    "Exported:",
    CUSTOMER_DIR / "customer_activation_master.csv"
)

display(
    customer_activation_master[
        [
            "customer_id_clean",
            "phone_clean",
            "address_raw",
            "address_join_key_customer",
            "campaign_touch_count",
            "first_campaign_seen",
            "last_campaign_seen",
            "customer_reply_count",
            "activation_status",
            "eligible_for_future_campaign",
        ]
    ].head(20)
)

Customer records: 9,557
Matched SMS contacts: 9,488
Verified responders: 445
STOP requests: 574
Eligible for future campaigns: 8,538
Exported: ../data/cleaned/customer/customer_activation_master.csv


,customer_id_clean,phone_clean,address_raw,address_join_key_customer,campaign_touch_count,first_campaign_seen,last_campaign_seen,customer_reply_count,activation_status,eligible_for_future_campaign
0,1,9162129352,6710 37 Ave,6710_37TH_AVE,1,D1_A,D1_A,0.0,NO_RESPONSE,True
1,4,9162301496,6425 Somis Way,6425_SOMIS_WAY,2,D3,D6,0.0,NO_RESPONSE,True
2,5,9163292467,6100 48th Ave #2202,6100_48TH_AVE,2,D3,D3_2,0.0,NO_RESPONSE,True
3,6,2792720793,3826 25th Ave,3826_25TH_AVE,1,D1_A,D1_A,0.0,NO_RESPONSE,True
4,7,9162065582,7101 Gerber Rd #216 #8106,7101_GERBER_RD,2,D3,D6,0.0,NO_RESPONSE,True
5,8,9167045539,5677 23rd St,5677_23RD_ST,2,D3,D6,0.0,NO_RESPONSE,True
6,13,9169527365,6267 Mlk Jr Blvd Apt 122,6267_MLK_JR_BLVD,2,D3,D4,0.0,NO_RESPONSE,True
7,15,9168064845,7809 36th Ave,7809_36TH_AVE,2,D3,D5,0.0,NO_RESPONSE,True
8,16,9168217817,4709 47th St,4709_47TH_ST,1,D2,D2,0.0,NO_RESPONSE,True
9,20,2792486208,6448 Stockton Blvd Room 27,6448_STOCKTON_BLVD,2,D3,D6,0.0,NO_RESPONSE,True


## 16.26 SMS Activation KPIs

With the Customer Activation Master completed, this section summarizes the overall performance of the SMS customer activation campaign.

These KPIs provide a high-level executive view of customer reach, engagement, opt-outs, and future marketing eligibility. The resulting summary will support the Tableau dashboard and executive reporting in later workbook sections.

In [98]:
activation = pd.read_csv(
    CUSTOMER_DIR / "customer_activation_master.csv"
)

total_customers = len(activation)

matched_sms_contacts = (
    activation["campaign_touch_count"] > 0
).sum()

verified_customers = (
    activation["activation_status"] == "VERIFIED_RESPONSE"
).sum()

stop_requests = (
    activation["activation_status"] == "STOP"
).sum()

no_response = (
    activation["activation_status"] == "NO_RESPONSE"
).sum()

eligible_customers = (
    activation["eligible_for_future_campaign"]
).sum()

response_rate = (
    verified_customers / matched_sms_contacts
)

stop_rate = (
    stop_requests / matched_sms_contacts
)

summary = pd.DataFrame({
    "metric": [
        "Total Customers",
        "Matched SMS Contacts",
        "Verified Responders",
        "No Response",
        "STOP Requests",
        "Eligible Customers",
        "Response Rate (%)",
        "STOP Rate (%)"
    ],
    "value": [
        total_customers,
        matched_sms_contacts,
        verified_customers,
        no_response,
        stop_requests,
        eligible_customers,
        round(response_rate * 100, 2),
        round(stop_rate * 100, 2)
    ]
})

summary.to_csv(
    CAMPAIGN_DIR / "sms_activation_summary.csv",
    index=False
)

print(f"Total customers: {total_customers:,}")
print(f"Matched SMS contacts: {matched_sms_contacts:,}")
print(f"Verified responders: {verified_customers:,}")
print(f"STOP requests: {stop_requests:,}")
print(f"Eligible for future campaigns: {eligible_customers:,}")
print(f"Exported: {CAMPAIGN_DIR / 'sms_activation_summary.csv'}")

display(summary)

Total customers: 9,557
Matched SMS contacts: 9,488
Verified responders: 445
STOP requests: 574
Eligible for future campaigns: 8,538
Exported: ../data/cleaned/campaigns/sms_activation_summary.csv


,metric,value
0,Total Customers,9557.00
1,Matched SMS Contacts,9488.00
2,Verified Responders,445.00
3,No Response,8469.00
4,STOP Requests,574.00
5,Eligible Customers,8538.00
6,Response Rate (%),4.69
7,STOP Rate (%),6.05


## 16.27 Campaign Performance Analysis

This section summarizes customer engagement by SMS campaign.

For each campaign, key marketing metrics including customer reach, verified responses, STOP requests, and response rates are calculated. These campaign-level KPIs provide a direct comparison of campaign effectiveness and identify which customer segments generated the strongest engagement.

The resulting summary supports executive reporting and Tableau visualizations.

In [99]:
activation = pd.read_csv(
    CUSTOMER_DIR / "customer_activation_master.csv"
)

campaigns = [
    "D1_A",
    "D1_B",
    "D2",
    "D3",
    "D3_2",
    "D4",
    "D5",
    "D6",
]

campaign_summary = []

for campaign in campaigns:

    campaign_customers = activation[
        (activation["first_campaign_seen"] == campaign)
        |
        (activation["last_campaign_seen"] == campaign)
    ].copy()

    reached = len(campaign_customers)

    verified = (
        campaign_customers["activation_status"]
        == "VERIFIED_RESPONSE"
    ).sum()

    stop = (
        campaign_customers["activation_status"]
        == "STOP"
    ).sum()

    eligible = (
        campaign_customers["eligible_for_future_campaign"]
    ).sum()

    response_rate = (
        verified / reached
        if reached > 0 else 0
    )

    stop_rate = (
        stop / reached
        if reached > 0 else 0
    )

    campaign_summary.append({
        "campaign": campaign,
        "customers_reached": reached,
        "verified_responders": verified,
        "stop_requests": stop,
        "eligible_customers": eligible,
        "response_rate_pct": round(response_rate * 100, 2),
        "stop_rate_pct": round(stop_rate * 100, 2),
    })

campaign_summary = pd.DataFrame(campaign_summary)

campaign_summary.to_csv(
    CAMPAIGN_DIR / "campaign_performance_summary.csv",
    index=False
)

print(f"Campaigns analyzed: {len(campaign_summary)}")
print(
    f"Exported: "
    f"{CAMPAIGN_DIR / 'campaign_performance_summary.csv'}"
)

display(campaign_summary)

Campaigns analyzed: 8
Exported: ../data/cleaned/campaigns/campaign_performance_summary.csv


,campaign,customers_reached,verified_responders,stop_requests,eligible_customers,response_rate_pct,stop_rate_pct
0,D1_A,509,27,40,442,5.30,7.86
1,D1_B,501,28,31,442,5.59,6.19
2,D2,3023,126,192,2705,4.17,6.35
3,D3,5469,264,313,4892,4.83,5.72
4,D3_2,987,50,54,883,5.07,5.47
5,D4,1507,74,86,1347,4.91,5.71
6,D5,1535,78,97,1360,5.08,6.32
7,D6,1538,68,80,1390,4.42,5.20


 ## 16.28 Campaign Touch Analysis

This section evaluates customer engagement based on the number of SMS campaign touches received.

Customers are grouped by campaign_touch_count to determine whether repeated campaign exposure influenced verified responses, STOP requests, and future marketing eligibility.

This analysis provides insight into customer engagement patterns and supports future campaign frequency planning.

In [104]:
activation = pd.read_csv(
    CUSTOMER_DIR / "customer_activation_master.csv"
)

touch_summary = (
    activation
    .groupby("campaign_touch_count", as_index=False)
    .agg(
        customers=("customer_id_clean", "count"),
        verified_responders=(
            "activation_status",
            lambda x: (x == "VERIFIED_RESPONSE").sum()
        ),
        stop_requests=(
            "activation_status",
            lambda x: (x == "STOP").sum()
        ),
        eligible_customers=(
            "eligible_for_future_campaign",
            "sum"
        ),
    )
)

touch_summary["touch_group"] = (
    touch_summary["campaign_touch_count"]
    .astype(str)
    .replace({
        "0": "Not Reached",
        "1": "1 Touch",
        "2": "2 Touches",
        "3": "3 Touches",
    })
)

touch_summary["response_rate_pct"] = (
    touch_summary["verified_responders"]
    / touch_summary["customers"]
    * 100
).round(2)

touch_summary["stop_rate_pct"] = (
    touch_summary["stop_requests"]
    / touch_summary["customers"]
    * 100
).round(2)

touch_summary = touch_summary[
    [
        "touch_group",
        "campaign_touch_count",
        "customers",
        "verified_responders",
        "stop_requests",
        "eligible_customers",
        "response_rate_pct",
        "stop_rate_pct",
    ]
]

touch_summary.to_csv(
    CAMPAIGN_DIR / "campaign_touch_summary.csv",
    index=False
)

print(f"Touch groups analyzed: {len(touch_summary)}")

print(
    "Exported:",
    CAMPAIGN_DIR / "campaign_touch_summary.csv"
)

display(touch_summary)

Touch groups analyzed: 4
Exported: ../data/cleaned/campaigns/campaign_touch_summary.csv


,touch_group,campaign_touch_count,customers,verified_responders,stop_requests,eligible_customers,response_rate_pct,stop_rate_pct
0,Not Reached,0,69,0,0,69,0.00,0.00
1,1 Touch,1,3907,175,255,3477,4.48,6.53
2,2 Touches,2,5419,260,315,4844,4.80,5.81
3,3 Touches,3,162,10,4,148,6.17,2.47


## 16.29 Geographic Engagement Summary

This section summarizes customer engagement by normalized address.

Verified responses, remaining marketable customers, and campaign touch counts are aggregated by customer address to identify geographic concentrations of engagement.

Addresses are prioritized by verified customer responses, followed by remaining marketable customers and campaign touch count. This analysis supports future neighborhood marketing, direct-mail planning, and geographic visualization in Tableau.

The resulting dataset establishes the foundation for future ZIP code, USPS carrier route, and geographic marketing analyses.

In [105]:
activation = pd.read_csv(
    CUSTOMER_DIR / "customer_activation_master.csv"
)

geo_summary = (
    activation
    .groupby(
        [
            "address_join_key_customer",
            "address_raw",
        ],
        as_index=False
    )
    .agg(
        campaign_touch_count=(
            "campaign_touch_count",
            "max"
        ),
        verified_responses=(
            "activation_status",
            lambda x: (x == "VERIFIED_RESPONSE").sum()
        ),
        stop_requests=(
            "activation_status",
            lambda x: (x == "STOP").sum()
        ),
        remaining_marketable_customers=(
            "eligible_for_future_campaign",
            "sum"
        ),
    )
)

geo_summary = geo_summary.sort_values(
    [
        "verified_responses",
        "remaining_marketable_customers",
        "campaign_touch_count",
    ],
    ascending=False
)

geo_summary.to_csv(
    CAMPAIGN_DIR / "geographic_engagement_summary.csv",
    index=False
)

print(f"Addresses analyzed: {len(geo_summary):,}")

print(
    "Exported:",
    CAMPAIGN_DIR / "geographic_engagement_summary.csv"
)

display(geo_summary.head(25))

Addresses analyzed: 6,890
Exported: ../data/cleaned/campaigns/geographic_engagement_summary.csv


,address_join_key_customer,address_raw,campaign_touch_count,verified_responses,stop_requests,remaining_marketable_customers
254,2301_53_AVE,2301 53 Ave,2,3,0,0
3688,5470_80_ST,5470 80 St,2,2,0,2
2745,4920_BAKER_AVE,4920 Baker Ave,2,2,0,1
3819,5551_JANSEN_DR,5551 Jansen Dr,2,2,0,1
5752,6812_CUNNINGHAM_WAY,6812 Cunningham Way,2,2,0,1
6587,7831_LEMON_HILL_AVE,7831 Lemon Hill Ave,2,2,0,1
5418,6501_ELDER_CREEK_RD,6501 Elder Creek Rd #30,3,2,0,0
1418,3972_67TH_ST,3972 67 St,2,2,0,0
5048,6213_BOOTH_LN,6213 Booth Lane,2,2,0,0
6760,8200_ELDER_CREEK_RD,8200 Elder Creek Rd,2,1,1,17


## 16.30 Executive Marketing Insights

This section summarizes the primary business insights discovered throughout the SMS customer activation analysis.

Executive KPIs, campaign performance, customer engagement, campaign touch analysis, and geographic engagement are consolidated into a concise executive report.

The report concludes with both the highest-performing campaign metrics and the recommended marketing strategy supported by the analysis.

In [107]:
activation = pd.read_csv(
    CUSTOMER_DIR / "customer_activation_master.csv"
)

campaign_summary = pd.read_csv(
    CAMPAIGN_DIR / "campaign_performance_summary.csv"
)

touch_summary = pd.read_csv(
    CAMPAIGN_DIR / "campaign_touch_summary.csv"
)

geo_summary = pd.read_csv(
    CAMPAIGN_DIR / "geographic_engagement_summary.csv"
)

best_campaign = campaign_summary.loc[
    campaign_summary["response_rate_pct"].idxmax()
]

best_touch = touch_summary.loc[
    touch_summary["response_rate_pct"].idxmax()
]

recommended_touch = touch_summary[
    touch_summary["touch_group"] == "2 Touches"
].iloc[0]

top_address = geo_summary.loc[
    geo_summary["verified_responses"].idxmax()
]

print("=" * 64)
print("PIZZA HOUSE SMS ACTIVATION ANALYSIS")
print("=" * 64)

print(f"\nMarketing-ready customers: {len(activation):,}")

print(
    f"Eligible customers: "
    f"{activation['eligible_for_future_campaign'].sum():,}"
)

print(
    f"Verified responders: "
    f"{(activation['activation_status'] == 'VERIFIED_RESPONSE').sum():,}"
)

print(
    f"STOP requests: "
    f"{(activation['activation_status'] == 'STOP').sum():,}"
)

print("\nTop Campaign")

print(
    f"  {best_campaign['campaign']} "
    f"({best_campaign['response_rate_pct']:.2f}% response rate)"
)

print("\nHighest Observed Touch Group")

print(
    f"  {best_touch['touch_group']} "
    f"({best_touch['response_rate_pct']:.2f}% response rate)"
)

print("\nRecommended Marketing Strategy")

print(
    f"  {recommended_touch['touch_group']}"
)

print(
    f"  Largest customer cohort ({recommended_touch['customers']:,} customers)"
)

print(
    f"  {recommended_touch['response_rate_pct']:.2f}% response rate"
)

print(
    f"  {recommended_touch['stop_rate_pct']:.2f}% STOP rate"
)

print("\nTop Performing Address")

print(
    f"  {top_address['address_raw']}"
)

print(
    f"  Verified Responses: "
    f"{top_address['verified_responses']}"
)

print("\nWorkbook 16 complete.")

PIZZA HOUSE SMS ACTIVATION ANALYSIS

Marketing-ready customers: 9,557
Eligible customers: 8,538
Verified responders: 445
STOP requests: 574

Top Campaign
  D1_B (5.59% response rate)

Highest Observed Touch Group
  3 Touches (6.17% response rate)

Recommended Marketing Strategy
  2 Touches
  Largest customer cohort (5,419 customers)
  4.80% response rate
  5.81% STOP rate

Top Performing Address
  2301 53 Ave
  Verified Responses: 3

Workbook 16 complete.
